# Update Field Collection Layer with New Detections

This notebook appends **new deep-learning detections** (e.g. from the Detect Objects or QA/QC workflow) into an existing **Field Collection layer** created by the companion notebook.

1. **Connect** to ArcGIS Online
2. **Load** the target field collection layer and the source detection layer
3. **Select schema** to map inherited fields (e.g. `radius_m` → `canopy_width_m`)
4. **Deduplicate** — skip features whose centroids are within a configurable distance of existing features
5. **Append** non-duplicate features with empty attributes
6. **Summary** — report added/skipped counts

### Deduplication Logic

For each new feature, its centroid is compared against all existing feature centroids. If any existing centroid is closer than the **dedup distance threshold**, the new feature is considered a duplicate and skipped. This prevents overlapping detections (e.g. trees detected in multiple runs or at different resolutions) from being added twice.

> **Requirements**: conda env `arcgis-ai`, Python 3.11, `arcgis` >= 2.4.


## Step 1 – Connect to ArcGIS Online

In [ ]:
import getpass, urllib3
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Staging toggle ─────────────────────────────────────────────────────
# Set to True for test runs — reads from the staging registry.
STAGING = True
# ───────────────────────────────────────────────────────────────────────

username = input("ArcGIS Online username: ")
password = getpass.getpass("ArcGIS Online password: ")
gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Signed in as: {gis.users.me.username} ({gis.properties.portalName})")
if STAGING:
    print("⚠️  STAGING MODE")

Setting `verify_cert` to False is a security risk, use at your own risk.


Signed in as: ralouta.aiddev (ArcGIS Online)


## Step 2 – Load Target & Source Layers

- **Target**: The field collection layer (auto-loaded from `item_registry.json` if available, otherwise manual input)
- **Source**: New detections to append (e.g. QA/QC tree points or raw detect objects output)

In [ ]:
import json
from pathlib import Path

# Auto-load target item ID from the appropriate registry
if STAGING:
    _registry_path = Path("..") / "item_registry_staging.json"
else:
    _registry_path = Path("..") / "item_registry.json"

target_item_id = None

if _registry_path.exists():
    _reg = json.loads(_registry_path.read_text())
    target_item_id = _reg.get("base_layer", {}).get("item_id")
    if target_item_id:
        print(f"Auto-loaded base layer ID from {_registry_path.name}: {target_item_id}")

if not target_item_id:
    target_item_id = input("Target layer Item ID (base layer or Field Collector view): ").strip()

source_item_id = input("Source detection layer Item ID: ").strip()

target_item = gis.content.get(target_item_id)
source_item = gis.content.get(source_item_id)

if target_item is None:
    raise ValueError(f"Target item '{target_item_id}' not found.")
if source_item is None:
    raise ValueError(f"Source item '{source_item_id}' not found.")

# Detect if target is a view layer — edits to an editable view flow through to the base layer
_is_view = (
    target_item.type == "Feature Service"
    and target_item.typeKeywords
    and "View Service" in target_item.typeKeywords
)
if _is_view:
    print("ℹ Target is a View Layer — edits will flow through to the base layer.")

target_layer = target_item.layers[0]
source_layer = source_item.layers[0]

# Query existing features (for dedup) and new features
# Reproject both to the target layer's SR for consistent distance calculations
target_sr = dict(target_layer.properties.extent.spatialReference)
target_wkid = target_sr.get("latestWkid", target_sr.get("wkid", 4326))

existing_fs = target_layer.query(where="1=1", out_sr=target_wkid)
source_fs = source_layer.query(where="1=1", out_sr=target_wkid)

print(f"Target: {target_item.title}{' (View)' if _is_view else ''}")
print(f"  Existing features: {len(existing_fs.features)}")
print(f"  Geometry type:     {target_layer.properties.geometryType}")
print(f"  SR WKID:           {target_wkid}")
print(f"\nSource: {source_item.title}")
print(f"  New features:      {len(source_fs.features)}")
print(f"  Geometry type:     {source_layer.properties.geometryType}")

Auto-loaded base layer ID from registry: ba63860ddf264ff0910ae5cffaab6500
Target: Plant Identification - AI
  Existing features: 2617
  Geometry type:     esriGeometryPoint
  SR WKID:           3857

Source: SAM3 Trees Voorburg 20 cm QAQC
  New features:      994
  Geometry type:     esriGeometryPoint


In [3]:
import ipywidgets as widgets
from IPython.display import display

if source_fs.features:
    # Build field list from source layer (exclude system/geometry fields)
    _SYS = {"objectid", "fid", "globalid", "shape", "shape_length", "shape_area",
            "shape__length", "shape__area", "st_length(shape)", "st_area(shape)"}
    _src_fields = [
        f for f in source_layer.properties.fields
        if f["name"].lower() not in _SYS
    ]
    _field_names = [f["name"] for f in _src_fields]

    _filter_enabled_w = widgets.Checkbox(value=False, description="Enable filter", indent=False)

    _field_w = widgets.Dropdown(
        options=_field_names,
        description="Field:",
        layout=widgets.Layout(width="350px"),
    )

    _op_w = widgets.Dropdown(
        options=[
            ("equals (=)", "="),
            ("not equal (<>)", "<>"),
            ("less than (<)", "<"),
            ("less or equal (<=)", "<="),
            ("greater than (>)", ">"),
            ("greater or equal (>=)", ">="),
            ("LIKE", "LIKE"),
            ("IS NULL", "IS NULL"),
            ("IS NOT NULL", "IS NOT NULL"),
        ],
        value="=",
        description="Operator:",
        layout=widgets.Layout(width="350px"),
    )

    _val_w = widgets.Text(
        value="",
        placeholder="Value (ignored for IS NULL / IS NOT NULL)",
        description="Value:",
        layout=widgets.Layout(width="400px"),
    )

    _filter_box = widgets.VBox([_field_w, _op_w, _val_w])
    _filter_box.layout.display = "none"

    def _toggle_filter(change):
        _filter_box.layout.display = "" if change["new"] else "none"
    _filter_enabled_w.observe(_toggle_filter, names="value")

    print("Optional: filter source features before dedup & append.")
    display(_filter_enabled_w)
    display(_filter_box)
    print("Run the next cell to apply the filter.")
else:
    _filter_enabled_w = None
    print("No source features — filter step skipped.")

Optional: filter source features before dedup & append.


Checkbox(value=False, description='Enable filter', indent=False)

Run the next cell to apply the filter.


In [4]:
# Apply the filter to source features (if enabled)
_source_features = list(source_fs.features)

if _source_features and _filter_enabled_w and _filter_enabled_w.value:
    _fname = _field_w.value
    _op = _op_w.value
    _val = _val_w.value.strip()

    _before = len(_source_features)

    def _matches(feat):
        v = (feat.attributes or {}).get(_fname)
        if _op == "IS NULL":
            return v is None
        if _op == "IS NOT NULL":
            return v is not None
        if v is None:
            return False
        # Try numeric comparison first
        try:
            v_num = float(v)
            val_num = float(_val)
            if _op == "=":   return v_num == val_num
            if _op == "<>":  return v_num != val_num
            if _op == "<":   return v_num < val_num
            if _op == "<=":  return v_num <= val_num
            if _op == ">":   return v_num > val_num
            if _op == ">=":  return v_num >= val_num
        except (ValueError, TypeError):
            pass
        # String comparison
        v_str = str(v)
        if _op == "=":    return v_str == _val
        if _op == "<>":   return v_str != _val
        if _op == "LIKE":
            import re
            pattern = "^" + re.escape(_val).replace(r"\%", ".*").replace(r"\_", ".") + "$"
            return bool(re.match(pattern, v_str, re.IGNORECASE))
        return False

    _source_features = [f for f in _source_features if _matches(f)]
    print(f"Filter: {_fname} {_op} {_val if _op not in ('IS NULL', 'IS NOT NULL') else ''}")
    print(f"  Before: {_before}  →  After: {len(_source_features)}")
else:
    if _source_features:
        print(f"No filter applied — using all {len(_source_features)} source features.")
    else:
        print("No source features — nothing to filter.")

Filter: det_type = circular
  Before: 994  →  After: 749


## Step 3 – Select Schema & Configure Dedup Distance

Select the schema to determine field mappings (e.g. `radius_m` → `canopy_width_m` for trees). Set the **deduplication distance** — new features whose centroid is closer than this threshold to any existing feature's centroid will be skipped.

In [5]:
import sys, os, importlib
import ipywidgets as widgets
from IPython.display import display

_repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import schemas
importlib.reload(schemas)

_schema_dropdown = widgets.Dropdown(
    options=list(schemas.SCHEMAS.keys()),
    value=list(schemas.SCHEMAS.keys())[0],
    description="Schema:",
    layout=widgets.Layout(width="350px"),
)

_dedup_dist_w = widgets.FloatText(
    value=2.0,
    description="Dedup dist (m):",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="250px"),
)

_dedup_help = widgets.HTML(
    value="<i>New features closer than this distance to an existing feature are skipped as duplicates. "
          "Set to 0 to disable deduplication.</i>"
)

display(_schema_dropdown)
display(widgets.HBox([_dedup_dist_w, _dedup_help]))
print("\nAdjust settings above, then run the next cell.")

Dropdown(description='Schema:', layout=Layout(width='350px'), options=('Plant Identification',), value='Plant …


Adjust settings above, then run the next cell.


In [6]:
# Load selected schema
mod_path = schemas.SCHEMAS[_schema_dropdown.value]
schema_mod = importlib.import_module(mod_path)
importlib.reload(schema_mod)

schema_label = _schema_dropdown.value
dedup_distance = _dedup_dist_w.value

print(f"Schema: {schema_label}")
print(f"Dedup distance: {dedup_distance} m")
if dedup_distance <= 0:
    print("  Deduplication DISABLED — all features will be appended.")

Schema: Plant Identification
Dedup distance: 2.0 m


## Step 4 – Deduplicate & Append

For each new feature:
1. Compute its **centroid** (for polygons) or use its point geometry directly
2. Check distance to all existing centroids — if any is within the dedup threshold, **skip**
3. Otherwise, build the feature with empty attributes and **append**

Uses a spatial index (sorted grid) for efficient distance checks when feature counts are large.


In [7]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from qaqc.dedup import centroid, deduplicate_features

# ── Compute existing centroids ──────────────────────────────────────
existing_centroids = [centroid(f.geometry) for f in existing_fs.features]

# ── Deduplicate (uses filtered source features) ────────────────────
print(f"Source features to process: {len(_source_features)}")
unique_source, duplicates_source, dedup_idx = deduplicate_features(
    new_features=_source_features,
    existing_centroids=existing_centroids,
    distance_m=dedup_distance,
    wkid=target_wkid,
)

# ── Build plain feature dicts (bypass Feature serialization issues) ──
# Features are appended with empty attributes.
features_to_add = []
for feat in unique_source:
    geom = feat.geometry
    _x = getattr(geom, "x", None)
    if _x is None and isinstance(geom, dict):
        _x = geom.get("x")
    _y = getattr(geom, "y", None)
    if _y is None and isinstance(geom, dict):
        _y = geom.get("y")

    if _x is not None and _y is not None:
        pt = {"x": float(_x), "y": float(_y),
              "spatialReference": {"wkid": target_wkid}}
    else:
        g = geom if isinstance(geom, dict) else dict(geom)
        cx, cy = centroid(g)
        pt = {"x": float(cx), "y": float(cy),
              "spatialReference": {"wkid": target_wkid}}

    features_to_add.append({"geometry": pt, "attributes": {}})

print(f"\nDeduplication results:")
print(f"  {dedup_idx.summary()}")
print(f"  Unique to add: {len(features_to_add)}")
print(f"  Duplicates:    {len(duplicates_source)}")


Source features to process: 749
Projected SR (WKID 3857): dedup distance = 2.0 map units

Deduplication results:
  Checked: 749  |  Duplicates: 195  |  Unique: 554
  Unique to add: 554
  Duplicates:    195


In [8]:
# ── Append features in batches via direct REST POST ────────────────
# Bypasses Feature object serialization which can inject extra geometry keys
import json

if features_to_add:
    BATCH = 500
    total_added = 0
    total_to_add = len(features_to_add)
    edit_url = f"{target_layer.url}/addFeatures"

    print(f"Appending {total_to_add} features to {target_item.title}...")
    for i in range(0, total_to_add, BATCH):
        batch = features_to_add[i : i + BATCH]
        params = {"f": "json", "features": json.dumps(batch)}
        result = gis._con.post(edit_url, params)
        added = sum(1 for r in result.get("addResults", []) if r.get("success"))
        total_added += added
        print(f"  Batch {i // BATCH + 1}: {added}/{len(batch)} added")

    print(f"\nTotal features added: {total_added}/{total_to_add}")
else:
    print("No new features to add (all duplicates or empty source).")

Appending 554 features to Plant Identification - AI...
  Batch 1: 500/500 added
  Batch 2: 54/54 added

Total features added: 554/554


In [9]:
# Recalculate and update the service extent after adding new features
from arcgis.features import FeatureLayerCollection

ext_result = target_layer.query(return_extent_only=True, out_sr=4326)
new_extent = ext_result["extent"]
new_extent["spatialReference"] = {"wkid": 4326}

flc = FeatureLayerCollection.fromitem(target_item)
flc.manager.update_definition({
    "initialExtent": new_extent,
    "fullExtent": new_extent,
})
print("Service extent updated to match current features.")

Service extent updated to match current features.


## Step 5 – Summary & Map

In [10]:
# Refresh count
final_count = target_layer.query(return_count_only=True)

print(f"Target layer: {target_item.title}")
print(f"  Item ID:          {target_item.itemid}")
print(f"  Features before:  {len(existing_fs.features)}")
print(f"  Duplicates:       {len(duplicates_source)}")
print(f"  New added:        {len(features_to_add)}")
print(f"  Features after:   {final_count}")
print(f"  Dedup distance:   {dedup_distance} m")
print(f"  Schema:           {schema_label}")

result_map = gis.map()
result_map.basemap.basemap = "satellite"
result_map.content.add(target_item)
result_map.extent = new_extent
result_map

Target layer: Plant Identification - AI
  Item ID:          ba63860ddf264ff0910ae5cffaab6500
  Features before:  2617
  Duplicates:       195
  New added:        554
  Features after:   3171
  Dedup distance:   2.0 m
  Schema:           Plant Identification


Map()